# Execute Recipes Programatically without database

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


In [ ]:
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthRequest
from femtum_sdk.api import HardwareApi

hardware_api = HardwareApi(host="localhost:5210")
hardware_api.__enter__()

hardware_health_response = hardware_api.health.GetHealth(HealthRequest())

print("Hardware Health:")
print(hardware_health_response)


## Execute Single Point Recipe

In [ ]:
from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import Attribute, Geometry, Point, Shape
from femtum_sdk.api.contracts.v1.dto.measure_pb2 import EnergyMeasure, EnergyMeasureUnit
from femtum_sdk.api.contracts.v1.dto.recipe_pb2 import LaserParameter, Recipe
from femtum_sdk.api.contracts.v1.hardware.laser_execution_pb2 import ExecuteLaserExecutionRecipeRequest, ExecuteLaserExecutionRecipeResponse, LaserExecutionFeatures, LaserExecutionOptions

result: ExecuteLaserExecutionRecipeResponse = hardware_api.laser_execution.ExecuteRecipe(ExecuteLaserExecutionRecipeRequest(
    Features=LaserExecutionFeatures(
        TakePictures=True,
    ),
    Options=LaserExecutionOptions(
        # CalibrationId="",
        EnergyPerPulse=EnergyMeasure(
            Value=0.5,
            Unit=EnergyMeasureUnit.Nanojoules,
        ),
    ),
    # ComponentPath="MyComponent",
    Recipe=Recipe(
        Geometries=[
            Geometry(
                Type="laser_marking",
                Attributes=[
                    Attribute(
                        Key="marking_type",
                        Value="single_pulse",
                    )
                ],
                Shape=Shape(
                    Point=Point(
                        X=100,
                        Y=100,
                    )
                ),
            )
        ],
        LaserParameters=[
            LaserParameter(Key="single_pulse_laser_triggering_frequency_khz", Value="2732", Unit="kHz"),
        ]
    )))

execution = result.Execution
print(execution)

## Cleanup

In [ ]:
hardware_api.__exit__()
